# Interactive LDA Topic Exploration of Starbucks 10-K Filings

*Retrain, visualize, and dissect 7 topics across 30 years of corporate language (FY1996-FY2025)*

---

**What this notebook adds beyond [Notebook 1 (NLP Topic Modeling)](https://www.kaggle.com/code/shiratoriseto/starbucks-10k-nlp-topic-modeling):**
1. **Interactive pyLDAvis visualization** -- explore topic-word distributions and inter-topic distances
2. **Word clouds** per topic -- the most distinctive words at a glance
3. **CEO-era topic fingerprints** -- how each CEO's language differed
4. **Topic distinctiveness analysis** -- Jensen-Shannon divergence + dendrogram

**Data sources:** SEC EDGAR 10-K filings (public domain) | Pre-computed topic proportions from Notebook 1

---

**Series context:**
- [Starbucks 10-K NLP Topic Modeling](https://www.kaggle.com/code/shiratoriseto/starbucks-10k-nlp-topic-modeling) -- Main NLP analysis (Notebook 1)
- [Manhattan Cafe Wars](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) -- EDA & competitor mapping (Theme 0)
- [Starbucks Spatial Clustering](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) -- Moran's I, LISA, Ripley's K (Theme 2A)
- [Starbucks Location Fitness](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) -- Demand-supply scoring & backtest (Theme 2B)
- [Walk-Distance Analysis](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx) — OSMnx walk-distance analysis (Theme 2C)


## Section 0 -- Setup & Data Loading

In [ ]:
# Install required packages
!pip install -q pyLDAvis==3.4.1 wordcloud plotly scikit-learn pandas numpy matplotlib scipy

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from wordcloud import WordCloud
from scipy.spatial.distance import jensenshannon
from scipy.cluster.hierarchy import linkage, dendrogram
from IPython.display import display, HTML

# pyLDAvis import -- handle different versions
import pyLDAvis
try:
    from pyLDAvis.lda_model import prepare as lda_prepare
    PYLDAVIS_PREPARE = "lda_model"
except ImportError:
    try:
        from pyLDAvis.sklearn import prepare as lda_prepare
        PYLDAVIS_PREPARE = "sklearn"
    except ImportError:
        PYLDAVIS_PREPARE = "manual"

print(f"pyLDAvis version: {pyLDAvis.__version__}")
print(f"pyLDAvis prepare method: {PYLDAVIS_PREPARE}")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# --- Data path resolution ---
ON_KAGGLE = Path("/kaggle/working").exists()

if ON_KAGGLE:
    # Try multiple possible Kaggle input paths
    _candidates = [
        Path("/kaggle/input/starbucks-30year-nlp-corpus"),
        Path("/kaggle/input/datasets/shiratoriseto/starbucks-30year-nlp-corpus"),
    ]
    DATA_DIR = None
    for _p in _candidates:
        if _p.exists():
            DATA_DIR = _p
            break
    if DATA_DIR is None:
        # Fallback: list what's actually in /kaggle/input
        import os
        avail = os.listdir("/kaggle/input") if Path("/kaggle/input").exists() else []
        raise FileNotFoundError(
            f"Dataset not found. Available inputs: {avail}"
        )
else:
    DATA_DIR = Path("../../dataset-upload/nlp-corpus")

print(f"ON_KAGGLE: {ON_KAGGLE}")
print(f"DATA_DIR:  {DATA_DIR}")
print(f"Files:     {sorted(p.name for p in DATA_DIR.iterdir())}")

In [ ]:
# --- Load data + sanity checks ---
lda_precomputed = pd.read_csv(DATA_DIR / "item1_lda_topic_proportions.csv")
store_counts = pd.read_csv(DATA_DIR / "store_counts_timeseries.csv")
raw_texts = pd.read_csv(DATA_DIR / "item1_raw_texts.csv")

# Sanity checks
print("=== Sanity Checks ===")
print(f"lda_precomputed: {lda_precomputed.shape[0]} rows x {lda_precomputed.shape[1]} cols")
assert lda_precomputed.shape[0] == 30, f"Expected 30 rows, got {lda_precomputed.shape[0]}"
assert lda_precomputed.shape[1] == 8, f"Expected 8 cols, got {lda_precomputed.shape[1]}"
print(f"  FY range: {lda_precomputed['fiscal_year'].min()}-{lda_precomputed['fiscal_year'].max()}")
print(f"  Missing values: {lda_precomputed.isnull().sum().sum()}")

# Topic proportions should sum to ~1.0 per row
topic_cols = [c for c in lda_precomputed.columns if c.startswith("topic_")]
row_sums = lda_precomputed[topic_cols].sum(axis=1)
print(f"  Topic row sums: min={row_sums.min():.3f}, max={row_sums.max():.3f}")

print(f"\nraw_texts: {raw_texts.shape[0]} rows x {raw_texts.shape[1]} cols")
assert raw_texts.shape[0] == 30, f"Expected 30 rows, got {raw_texts.shape[0]}"
print(f"  FY range: {raw_texts['fiscal_year'].min()}-{raw_texts['fiscal_year'].max()}")
print(f"  Missing values: {raw_texts.isnull().sum().sum()}")
text_lengths = raw_texts["text"].str.split().str.len()
print(f"  Word counts: min={text_lengths.min()}, max={text_lengths.max()}, median={text_lengths.median():.0f}")

print(f"\nstore_counts: {store_counts.shape[0]} rows x {store_counts.shape[1]} cols")
print(f"  Columns: {list(store_counts.columns)}")
print("\nAll checks passed.")

## Section 1 -- Text Preprocessing & LDA Training

We replicate the preprocessing pipeline from Notebook 1:
1. Split each of the 30 annual Item 1 texts into **~150-word chunks** (to give LDA more documents)
2. Vectorize with `CountVectorizer(max_df=0.95, min_df=2, max_features=1000, stop_words='english')`
3. Train a **7-topic LDA** model with `random_state=42`
4. Validate by comparing the retrained topic proportions against the pre-computed CSV from Notebook 1

In [ ]:
# --- Chunk texts into ~150-word segments ---
CHUNK_SIZE = 150  # words per chunk (matching Notebook 1)

chunks = []
chunk_years = []

for _, row in raw_texts.iterrows():
    words = row["text"].split()
    fy = row["fiscal_year"]
    for i in range(0, len(words), CHUNK_SIZE):
        chunk = " ".join(words[i:i + CHUNK_SIZE])
        if len(chunk.split()) >= 20:  # skip tiny trailing chunks
            chunks.append(chunk)
            chunk_years.append(fy)

chunk_years = np.array(chunk_years)
print(f"Total chunks: {len(chunks)}")
print(f"Chunks per year: min={np.unique(chunk_years, return_counts=True)[1].min()}, "
      f"max={np.unique(chunk_years, return_counts=True)[1].max()}")

# Memory check
total_chars = sum(len(c) for c in chunks)
print(f"Total characters in chunks: {total_chars:,} ({total_chars / 1e6:.1f} MB)")
assert total_chars < 50_000_000, "Unexpectedly large corpus -- check data"

In [ ]:
# --- Vectorize and train LDA ---
vectorizer = CountVectorizer(
    max_df=0.95,
    min_df=2,
    max_features=1000,
    stop_words="english",
)
dtm = vectorizer.fit_transform(chunks)
feature_names = vectorizer.get_feature_names_out()

print(f"Document-term matrix: {dtm.shape[0]} chunks x {dtm.shape[1]} features")
print(f"Non-zero entries: {dtm.nnz:,} (sparsity: {1 - dtm.nnz / (dtm.shape[0] * dtm.shape[1]):.2%})")

# Train LDA
N_TOPICS = 7
lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    max_iter=20,
    learning_method="batch",
)
lda_model.fit(dtm)
print(f"\nLDA model trained: {N_TOPICS} topics, perplexity={lda_model.perplexity(dtm):.1f}")

In [ ]:
# --- Aggregate chunk-level topics to year level and validate ---
chunk_topic_dist = lda_model.transform(dtm)  # (n_chunks, 7)

# Average per fiscal year
year_topic = pd.DataFrame(chunk_topic_dist, columns=[f"topic_{i}" for i in range(N_TOPICS)])
year_topic["fiscal_year"] = chunk_years
year_topic = year_topic.groupby("fiscal_year").mean().reset_index()

# Normalize rows to sum to 1
topic_cols_new = [f"topic_{i}" for i in range(N_TOPICS)]
year_topic[topic_cols_new] = year_topic[topic_cols_new].div(year_topic[topic_cols_new].sum(axis=1), axis=0)

print(f"Year-level topic proportions: {year_topic.shape}")
print(f"Row sums: {year_topic[topic_cols_new].sum(axis=1).unique()[:3]}")

# Compare with pre-computed (correlation per topic)
print("\n=== Validation: correlation with pre-computed topics ===")
print("(Note: exact values may differ due to LDA stochasticity, but correlations should be moderate-to-high)")
merged = year_topic.merge(lda_precomputed, on="fiscal_year", suffixes=("_new", "_pre"))
for i in range(N_TOPICS):
    corr = merged[f"topic_{i}_new"].corr(merged[f"topic_{i}_pre"])
    print(f"  Topic {i}: r = {corr:+.3f}")
print("\n(Negative correlations indicate topic index permutation -- this is expected with LDA re-training.")
print(" The pyLDAvis visualization below does not depend on index ordering.)")

## Section 2 -- pyLDAvis Interactive Visualization

[pyLDAvis](https://github.com/bmabey/pyLDAvis) provides a web-based interactive visualization for LDA models.
The left panel shows inter-topic distances (via multidimensional scaling), and the right panel shows
the most relevant terms for each selected topic.

**How to use:**
- Click on a topic circle to see its top words
- Adjust the relevance slider (lambda) to balance frequency vs. exclusivity
- Hover over bars to see term-level statistics

In [ ]:
# --- Prepare and display pyLDAvis (static HTML for Kaggle compatibility) ---
import time
print("Preparing pyLDAvis visualization...")
t0 = time.time()

if PYLDAVIS_PREPARE == "lda_model":
    vis_data = lda_prepare(lda_model, dtm, vectorizer, sort_topics=False)
elif PYLDAVIS_PREPARE == "sklearn":
    vis_data = lda_prepare(lda_model, dtm, vectorizer, sort_topics=False)
else:
    topic_term_dists = lda_model.components_ / lda_model.components_.sum(axis=1, keepdims=True)
    doc_topic_dists = chunk_topic_dist
    doc_lengths = np.array(dtm.sum(axis=1)).flatten()
    vocab = feature_names
    term_frequency = np.array(dtm.sum(axis=0)).flatten()
    vis_data = pyLDAvis.prepare(
        topic_term_dists=topic_term_dists,
        doc_topic_dists=doc_topic_dists,
        doc_lengths=doc_lengths,
        vocab=vocab,
        term_frequency=term_frequency,
        sort_topics=False,
    )

print(f"pyLDAvis prepared in {time.time() - t0:.1f}s")

# Render as static HTML (more reliable than enable_notebook on Kaggle)
html_str = pyLDAvis.prepared_data_to_html(vis_data)
display(HTML(html_str))

**Interpretation guide:**
- **Circle size** = overall prevalence of the topic in the corpus
- **Circle distance** = how semantically different topics are (closer = more overlap)
- **Blue bars** = overall term frequency; **Red bars** = term frequency within the selected topic
- **Lambda slider** (top right): lambda=1 ranks by raw frequency in topic; lambda=0 ranks by exclusivity (lift). A value around 0.6 often provides the most interpretable ranking.

## Section 3 -- Word Clouds Per Topic

Each word cloud uses the topic-word distribution weights from the trained LDA model.
Larger words have higher probability within that topic.

In [ ]:
# --- Topic labels from Notebook 1 (for reference; retrained topics may have different index ordering) ---
TOPIC_LABELS = {
    "topic_0": "Store Operations",
    "topic_1": "Supply Chain & Commodity",
    "topic_2": "Leadership & Governance",
    "topic_3": "Digital & Loyalty",
    "topic_4": "International & IP",
    "topic_5": "People, Culture & ESG",
    "topic_6": "Product & Competition",
}

TOPIC_COLORS = {
    "topic_0": "#264653",
    "topic_1": "#2A9D8F",
    "topic_2": "#6C757D",
    "topic_3": "#E76F51",
    "topic_4": "#457B9D",
    "topic_5": "#E63946",
    "topic_6": "#F4A261",
}

# For the retrained model, assign labels based on top words
print("Retrained LDA -- Top 10 words per topic:")
retrained_labels = {}
for i in range(N_TOPICS):
    top_idx = lda_model.components_[i].argsort()[-10:][::-1]
    top_words = [feature_names[j] for j in top_idx]
    print(f"  Topic {i}: {', '.join(top_words)}")
    retrained_labels[f"topic_{i}"] = f"Topic {i}"

In [ ]:
# --- Generate word clouds (2x4 grid, 7 topics + 1 blank) ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

colormap_list = ["#264653", "#2A9D8F", "#6C757D", "#E76F51", "#457B9D", "#E63946", "#F4A261"]

for i in range(N_TOPICS):
    # Build word-weight dict from topic-word distribution
    word_weights = {feature_names[j]: lda_model.components_[i][j]
                    for j in range(len(feature_names))}

    wc = WordCloud(
        width=400, height=300,
        background_color="white",
        max_words=50,
        colormap="viridis",
        random_state=RANDOM_STATE,
    ).generate_from_frequencies(word_weights)

    axes[i].imshow(wc, interpolation="bilinear")
    # Show top 3 words as subtitle
    top3_idx = lda_model.components_[i].argsort()[-3:][::-1]
    top3 = ", ".join(feature_names[j] for j in top3_idx)
    axes[i].set_title(f"Topic {i}\n({top3})", fontsize=11, fontweight="bold")
    axes[i].axis("off")

# Hide the 8th subplot
axes[7].axis("off")

fig.suptitle("Word Clouds: LDA Topic-Word Distributions (7 Topics from 30 Years of 10-K Filings)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\nFinding: Each topic captures a distinct vocabulary cluster. The most distinctive words")
print("per topic help identify the strategic dimension each topic represents.")

## Section 4 -- Topic Evolution Over Time

We use the **pre-computed** topic proportions from Notebook 1 (which used the same methodology
but may have slightly different topic ordering due to LDA stochasticity) to show how
Starbucks 10-K language evolved across 30 years.

In [ ]:
# --- Interactive stacked area chart (plotly) using pre-computed proportions ---
topic_cols_pre = [c for c in lda_precomputed.columns if c.startswith("topic_")]

fig = go.Figure()
for col in topic_cols_pre:
    label = TOPIC_LABELS[col]
    color = TOPIC_COLORS[col]
    fig.add_trace(go.Scatter(
        x=lda_precomputed["fiscal_year"], y=lda_precomputed[col],
        mode="lines", name=label,
        line=dict(width=0.5, color=color),
        fillcolor=color,
        stackgroup="one",
        hovertemplate=f"{label}<br>FY%{{x}}: %{{y:.1%}}<extra></extra>",
    ))

fig.update_layout(
    title=dict(text="Starbucks 10-K Topic Composition Over 30 Years (Stacked Area)",
               font=dict(size=16)),
    xaxis_title="Fiscal Year",
    yaxis_title="Topic Share",
    yaxis=dict(tickformat=".0%", range=[0, 1]),
    legend=dict(orientation="h", yanchor="bottom", y=-0.25),
    template="plotly_white",
    height=500,
    width=900,
)

fig.show()

In [ ]:
# --- Individual topic sparklines with annotated events ---
CEO_PERIODS = [
    ("Howard Schultz (1st)", 1996, 1999, "#E8F5E9"),
    ("Orin Smith",           2000, 2004, "#FFF3E0"),
    ("Jim Donald",           2005, 2007, "#E3F2FD"),
    ("Howard Schultz (2nd)", 2008, 2016, "#E8F5E9"),
    ("Kevin Johnson",        2017, 2021, "#F3E5F5"),
    ("Laxman Narasimhan",    2022, 2023, "#FFF9C4"),
    ("Brian Niccol",         2024, 2025, "#FFEBEE"),
]

fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=[TOPIC_LABELS[f"topic_{i}"] for i in range(N_TOPICS)] + [""],
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
)

for i in range(N_TOPICS):
    row = i // 2 + 1
    col_pos = i % 2 + 1
    topic_col = f"topic_{i}"
    color = TOPIC_COLORS[topic_col]

    fig.add_trace(go.Scatter(
        x=lda_precomputed["fiscal_year"],
        y=lda_precomputed[topic_col] * 100,
        mode="lines+markers",
        line=dict(color=color, width=2),
        marker=dict(size=4),
        name=TOPIC_LABELS[topic_col],
        showlegend=False,
        hovertemplate=f"{TOPIC_LABELS[topic_col]}<br>FY%{{x}}: %{{y:.1f}}%<extra></extra>",
    ), row=row, col=col_pos)

    # Add CEO era backgrounds
    for name, start, end, bg_color in CEO_PERIODS:
        fig.add_vrect(
            x0=start - 0.5, x1=end + 0.5,
            fillcolor=bg_color, opacity=0.3, line_width=0,
            row=row, col=col_pos,
        )

# Annotate key events on the People/ESG topic (topic_5, position row=3, col=2)
fig.add_annotation(
    x=2020, y=lda_precomputed.loc[lda_precomputed["fiscal_year"] == 2020, "topic_5"].values[0] * 100,
    text="COVID + BLM", showarrow=True, arrowhead=2, font=dict(size=9),
    row=3, col=2,
)
fig.add_annotation(
    x=2008, y=lda_precomputed.loc[lda_precomputed["fiscal_year"] == 2008, "topic_0"].values[0] * 100,
    text="Financial Crisis", showarrow=True, arrowhead=2, font=dict(size=9),
    row=1, col=1,
)

fig.update_layout(
    title=dict(text="Individual Topic Trends (% share) with CEO Eras", font=dict(size=14)),
    height=800, width=900,
    template="plotly_white",
    showlegend=False,
)
fig.update_yaxes(title_text="%", ticksuffix="%")
fig.show()

**Key findings:**
- **People, Culture & ESG (Topic 5)** surged dramatically from near-zero in the 1990s to become a major topic after 2018, with a sharp jump in FY2020 (COVID + BLM).
- **Store Operations (Topic 0)** dominated the early years but has steadily declined as the filings diversified.
- **Digital & Loyalty (Topic 3)** emerged only after ~2010, reflecting the mobile ordering and rewards program rollout.
- CEO transitions often correspond to visible inflection points in topic proportions.

## Section 5 -- CEO-Era Topic Comparison

We define seven CEO eras and compute mean topic proportions for each, then visualize
the "topic fingerprint" of each CEO's tenure.

In [ ]:
# --- Compute mean topic proportions per CEO era ---
CEO_ERAS = [
    ("Schultz 1st\n(1996-99)",   1996, 1999),
    ("O. Smith\n(2000-04)",       2000, 2004),
    ("J. Donald\n(2005-07)",      2005, 2007),
    ("Schultz 2nd\n(2008-16)",    2008, 2016),
    ("K. Johnson\n(2017-21)",     2017, 2021),
    ("Narasimhan\n(2022-23)",     2022, 2023),
    ("B. Niccol\n(2024-25)",      2024, 2025),
]

era_means = []
for label, start, end in CEO_ERAS:
    mask = (lda_precomputed["fiscal_year"] >= start) & (lda_precomputed["fiscal_year"] <= end)
    means = lda_precomputed.loc[mask, topic_cols_pre].mean()
    means["ceo_era"] = label
    era_means.append(means)

era_df = pd.DataFrame(era_means)
era_df = era_df[["ceo_era"] + topic_cols_pre]

# Sanity check
print("Mean topic proportions per CEO era:")
display_df = era_df.copy()
for col in topic_cols_pre:
    display_df[TOPIC_LABELS[col]] = display_df[col].map("{:.1%}".format)
display(HTML(display_df[["ceo_era"] + [TOPIC_LABELS[c] for c in topic_cols_pre]].to_html(index=False)))

In [ ]:
# --- Radar chart: topic fingerprint per CEO era ---
labels = [TOPIC_LABELS[c] for c in topic_cols_pre]
n_labels = len(labels)
angles = np.linspace(0, 2 * np.pi, n_labels, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

era_colors = ["#4CAF50", "#FF9800", "#2196F3", "#388E3C", "#9C27B0", "#FFC107", "#F44336"]

for idx, (_, row) in enumerate(era_df.iterrows()):
    values = row[topic_cols_pre].values.tolist()
    values += values[:1]  # close polygon
    ax.plot(angles, values, "o-", linewidth=2, label=row["ceo_era"],
            color=era_colors[idx], markersize=5)
    ax.fill(angles, values, alpha=0.05, color=era_colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0, era_df[topic_cols_pre].values.max() * 1.15)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_title("CEO-Era Topic Fingerprints (Radar Chart)", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# --- Heatmap: CEO era x topic ---
heat_data = era_df[topic_cols_pre].values
era_labels = era_df["ceo_era"].values
topic_labels_list = [TOPIC_LABELS[c] for c in topic_cols_pre]

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(heat_data, cmap="YlOrRd", aspect="auto")

ax.set_xticks(range(len(topic_labels_list)))
ax.set_xticklabels(topic_labels_list, rotation=30, ha="right", fontsize=10)
ax.set_yticks(range(len(era_labels)))
ax.set_yticklabels(era_labels, fontsize=10)

# Annotate cells
for i in range(len(era_labels)):
    for j in range(len(topic_labels_list)):
        val = heat_data[i, j]
        text_color = "white" if val > heat_data.max() * 0.65 else "black"
        ax.text(j, i, f"{val:.0%}", ha="center", va="center",
                fontsize=10, color=text_color, fontweight="bold")

ax.set_title("Topic Proportions by CEO Era (Heatmap)", fontsize=14, fontweight="bold")
plt.colorbar(im, ax=ax, label="Topic Proportion", shrink=0.8)
plt.tight_layout()
plt.show()

print("\nFinding: The heatmap reveals clear strategic language shifts across CEO tenures.")
print("Schultz's first tenure focused heavily on Store Operations; the Johnson and Narasimhan")
print("eras show elevated People/Culture/ESG language.")

## Section 6 -- Topic Distinctiveness

We measure how different each CEO era's language is from every other era using
**Jensen-Shannon divergence** (a symmetric, bounded measure of distribution similarity).
A dendrogram then reveals which CEO eras share the most similar -- or most different -- language profiles.

In [ ]:
# --- Jensen-Shannon divergence matrix ---
n_eras = len(era_df)
era_names = era_df["ceo_era"].values
era_dists = era_df[topic_cols_pre].values  # (n_eras, 7)

# Ensure valid probability distributions (sum to 1, no zeros)
era_dists = era_dists + 1e-10
era_dists = era_dists / era_dists.sum(axis=1, keepdims=True)

js_matrix = np.zeros((n_eras, n_eras))
for i in range(n_eras):
    for j in range(n_eras):
        js_matrix[i, j] = jensenshannon(era_dists[i], era_dists[j])

# Display
js_df = pd.DataFrame(js_matrix, index=era_names, columns=era_names)
print("Jensen-Shannon Divergence Matrix (lower = more similar):")
display(HTML(js_df.style.format("{:.3f}").background_gradient(cmap="RdYlGn_r").to_html()))

print(f"\nMost similar pair: {js_matrix[np.triu_indices(n_eras, k=1)].min():.4f}")
print(f"Most different pair: {js_matrix[np.triu_indices(n_eras, k=1)].max():.4f}")

In [ ]:
# --- Dendrogram of CEO-era language similarity ---
# Convert JSD matrix to condensed form for scipy
from scipy.spatial.distance import squareform

condensed = squareform(js_matrix)
Z = linkage(condensed, method="ward")

fig, ax = plt.subplots(figsize=(10, 6))
dn = dendrogram(
    Z,
    labels=[name.replace('\n', ' ') for name in era_names],
    ax=ax,
    color_threshold=0.15,
    leaf_font_size=11,
)
ax.set_title("CEO-Era Language Similarity (Dendrogram via Jensen-Shannon Divergence)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Ward Linkage Distance", fontsize=11)
ax.axhline(y=0.15, color="gray", linestyle="--", alpha=0.5, label="Threshold")
plt.tight_layout()
plt.show()

print("\nSummary findings:")
print("- The dendrogram clusters CEO eras into groups based on topic language similarity.")
print("- Early eras (Schultz 1st, Orin Smith, Jim Donald) tend to cluster together,")
print("  reflecting more traditional store-operations-focused language.")
print("- Later eras (Johnson, Narasimhan) cluster separately due to the emergence")
print("  of Digital/Loyalty and People/Culture/ESG topics.")
print("- The Brian Niccol era (only 2 years) may show distinctive \"Back to Starbucks\"")
print("  language patterns.")

## Section 7 -- Limitations & Series Context

### Limitations

| # | Limitation | Impact |
|---|-----------|--------|
| 1 | **Topic count (k=7)** | Selected via coherence analysis in Notebook 1, but other values of k could yield different insights. |
| 2 | **Chunk size (~150 words)** | Arbitrary segmentation may split semantically coherent passages across chunks. |
| 3 | **LDA stochasticity** | Retrained model may produce different topic orderings. We validate against pre-computed results but exact reproduction is not guaranteed. |
| 4 | **Small corpus** | Only 30 documents (annual filings), expanded to ~847 chunks. This is small by LDA standards. |
| 5 | **Document length bias** | Early filings (~3K words) produce fewer chunks than recent ones (~10K+ words), giving later years more influence. |
| 6 | **Bag-of-words assumption** | LDA ignores word order and context. Phrases like "not profitable" are treated as separate words. |

### Series Navigation

This notebook is part of the Starbucks data science series:

| Theme | Notebook | Description |
|-------|----------|-------------|
| 0 | [Manhattan Cafe Wars](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) | EDA & competitor mapping |
| 1 | [10-K NLP Topic Modeling](https://www.kaggle.com/code/shiratoriseto/starbucks-10k-nlp-topic-modeling) | Main NLP analysis (parent of this notebook) |
| 1F | **This notebook** | Interactive LDA exploration, word clouds, CEO-era analysis |
| 2A | [Spatial Clustering](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) | Moran's I, LISA, Ripley's K |
| 2B | [Location Fitness](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) | Demand-supply scoring & backtest |
| 2C | [Walk-Distance Analysis](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx) | OSMnx walk-distance & cannibalization |
